In [1]:
print("hello")

hello


In [3]:
import json
import pandas as pd

file_path = 'sec_finetune_v20.jsonl'

def extract_detailed_diagnostics(entry):
    meta = entry.get('instruction_pair', {}).get('metadata', {}) or entry.get('metadata', {})
    root_meta = entry.get('meta', {})
    score = meta.get('quality_score', 0)
    
    return {
        'id': entry.get('id', 'N/A'),
        'ticker': root_meta.get('ticker', 'N/A'),
        'form_type': root_meta.get('form_type', 'N/A'),
        'score': score,
        'metrics_count': meta.get('metrics_count', 0),
        'confidence': meta.get('evidence_confidence', 'N/A'),
        'hallucination': meta.get('hallucination_flag', False),
        'sanity_action': meta.get('sanity_action', 'N/A'),
        'text_filled': meta.get('text_sections_filled', 0),
        'xbrl_pct': meta.get('data_source_mix', {}).get('xbrl_pct', 0),
        'causal_density': meta.get('causal_density_score', 0)
    }

detailed_rejected = []
with open(file_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        score = (data.get('instruction_pair', {}).get('metadata', {}) or data.get('metadata', {})).get('quality_score')
        if score is not None and score < 80:
            detailed_rejected.append(extract_detailed_diagnostics(data))

df = pd.DataFrame(detailed_rejected)
df.to_csv('auditor_rejections_detailed_v20.csv', index=False)

# Analysis prints
print(df.groupby('form_type')['metrics_count'].mean())

# Display the top drivers of low scores
print(df.head(15).to_markdown(index=False))

form_type
10-K      35.500000
10-Q      35.666667
10-Q/A    39.000000
Name: metrics_count, dtype: float64
| id           | ticker   | form_type   |   score |   metrics_count | confidence   | hallucination   | sanity_action   |   text_filled |   xbrl_pct |   causal_density |
|:-------------|:---------|:------------|--------:|----------------:|:-------------|:----------------|:----------------|--------------:|-----------:|-----------------:|
| 89ba6c5f48ee | VTRS     | 10-K        |      75 |              35 | LOW          | False           | pass            |             2 |        100 |               54 |
| c67f003f186b | RCL      | 10-Q        |      75 |              34 | LOW          | False           | pass            |             2 |        100 |               44 |
| 4d9c1127b2fe | YUM      | 10-Q/A      |      70 |              39 | LOW          | False           | pass            |             2 |        100 |               17 |
| b1edbaa32a96 | T        | 10-Q        |      75

In [1]:
import json
import pandas as pd
from collections import Counter
from tabulate import tabulate

# Configuration
file_path = 'sec_finetune_v20.jsonl'

def find_quality_score(obj):
    """Recursively search for quality_score in a dictionary/list structure."""
    if isinstance(obj, dict):
        if 'quality_score' in obj:
            return obj['quality_score']
        for v in obj.values():
            res = find_quality_score(v)
            if res is not None:
                return res
    elif isinstance(obj, list):
        for item in obj:
            res = find_quality_score(item)
            if res is not None:
                return res
    return None

# Extraction
scores = []
with open(file_path, 'r') as f:
    for line in f:
        try:
            entry = json.loads(line)
            score = find_quality_score(entry)
            if score is not None:
                scores.append(score)
        except:
            continue

# Aggregation
if scores:
    total = len(scores)
    score_counts = Counter(scores)
    
    df_results = pd.DataFrame([
        {'Quality Score': k, 'Count': v, 'Proportion (%)': round((v / total) * 100, 2)}
        for k, v in score_counts.items()
    ])
    
    # Sorting by Score descending for better readability
    df_results = df_results.sort_values(by='Quality Score', ascending=False)
    
    # Save results
    df_results.to_csv('quality_score_analysis.csv', index=False)
    print(df_results.to_markdown(index=False))
else:
    print("No quality scores found.")

|   Quality Score |   Count |   Proportion (%) |
|----------------:|--------:|-----------------:|
|             100 |     607 |             60.7 |
|              90 |     287 |             28.7 |
|              85 |      39 |              3.9 |
|              80 |      61 |              6.1 |
|              75 |       5 |              0.5 |
|              70 |       1 |              0.1 |
